# M3 lab: what should be built? Capacity expansion with representative weeks, validated

*Energy Systems, Step by Step* · Modeler tier · lab for lesson M3 · about 60 minutes, plus 120 minutes of practice at the end.

You will rebuild the lesson's capacity-expansion linear program in Python, solve it with the open-source HiGHS solver through `scipy.optimize.linprog(method="highs")`, and check your own results against the lesson's reference block with `assert` cells:

1. Prairie Town's year: real 2025 hourly shapes, and 2024 as a second weather year.
2. Representative weeks: medoids, extreme weeks and weights.
3. The expansion LP: what to build and how to run it, with the battery linked across weeks.
4. The checks: the plan re-run hour by hour on all 8,760 hours of 2025 and of 2024, and the error measures.
5. Three rules for storage between weeks, and the full-year optimum.

**How to run it.** Put this notebook and the four data files in one folder. In Google Colab: *File → Upload notebook*, then upload the four files with the folder icon on the left, then *Runtime → Run all* (Colab has NumPy and SciPy). Locally: Python 3.9 or later with `numpy` and `scipy` 1.9 or later. The full-year optimum takes seconds to a minute.

| File (download from the lesson page) | What it holds |
|---|---|
| `m03-capacity-expansion.json` | the lesson's choices (cap, sets of weeks, storage what-if) and its reference results |
| `13-capstone.json` | lesson 13's costs, with their sources and dates (EIA, NREL) |
| `town_year.json` | the 8,760 hourly shapes of 2025 (Southwest Power Pool, EIA-930), in thousandths of each series' maximum |
| `m03-capacity-expansion-2024.json` | the same for 2024, with 29 February left out |

**The data.** Prairie Town is fictional. Its demand, wind and solar follow real hourly shapes of the Southwest Power Pool grid (EIA Form EIA-930); its costs are EIA's and NREL's, constant within the year. All costs are in 2025 dollars: NREL's 2024 battery costs are converted with BEA's GDP implicit price deflator (`deflator` in `13-capstone.json`). Results are illustrative, not recommendations.

**What the asserts check.** Your own weeks, LP sizes, estimates and sizes built; then your own plans re-run on both years: spending, CO₂ and unserved energy. Hours short, the weeks they fall in and the largest shortfall are printed, not asserted: other schedules of the same cost can spread a shortfall differently.

In [ ]:
import json
import time
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import scipy
from scipy.optimize import linprog
from scipy.sparse import coo_matrix

major, minor = (int(x) for x in scipy.__version__.split(".")[:2])
assert (major, minor) >= (1, 9), "this lab needs SciPy 1.9 or later (pip install -U scipy)"
FILES = ["m03-capacity-expansion.json", "13-capstone.json", "town_year.json", "m03-capacity-expansion-2024.json"]
for name in FILES:
    assert Path(name).exists(), f"{name} not found: put it next to this notebook (in Colab, upload it with the folder icon)."
D, cfg, year, year24 = (json.load(open(n)) for n in FILES)
REF = D["reference"]
print("NumPy", np.__version__, "· SciPy", scipy.__version__)
print(year["meta"]["source"])
print(year24["meta"]["title"])

## 1. The town's year

- **Demand** is the 2025 shape (thousandths of SPP's 2025 peak hour) times 450 MW.
- **Wind and solar** per MW built: each shape is scaled by a factor $k$ so that $\text{mean}(\min(1, k \cdot \text{shape}))$ equals EIA's 2025 U.S. capacity factor (wind 33.8%, solar 24.0%; EIA, *Electric Power Monthly*, Table 6.07.B). An hour is never above the plant's size.
- **2024**, the second weather year, uses the same rules, with demand scaled so that the year's energy equals 2025's. Only the hour-by-hour pattern differs.

In [ ]:
CF = {"wind": 0.338, "solar": 0.240}        # EIA, Electric Power Monthly, Table 6.07.B, 2025 annual, U.S. utility scale
HOURS, HPW, WEEKS = 8760, 168, 52

def cf_scale(shape, cf):
    # k with mean(min(1, k * s)) = cf, by bisection (the lesson's LX.townProfile, method "scale")
    s = np.maximum(np.asarray(shape, float), 0) / 1000
    f = lambda k: np.minimum(1, k * s).mean()
    lo, hi = 0.0, 1.0
    while f(hi) < cf:
        hi *= 2
    for _ in range(200):
        if hi - lo <= 1e-13 * max(1, hi):
            break
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if f(mid) < cf else (lo, mid)
    return (lo + hi) / 2

def prepare_year(y, peak_mw):
    out = {"demand": np.asarray(y["demand"], float) / 1000 * peak_mw, "start_utc": y["start_utc"]}
    for k in ("wind", "solar"):
        s = np.asarray(y[k], float) / 1000
        out[k] = np.minimum(1, cf_scale(y[k], CF[k]) * s)
    out["peak"] = out["demand"].max()
    return out

Y = prepare_year(year, cfg["town"]["demandPeakMW"])
Y24 = prepare_year(year24, Y["demand"].sum() / (np.asarray(year24["demand"], float) / 1000).sum())
for name, y in (("2025", Y), ("2024", Y24)):
    print(f"{name}: demand {y['demand'].sum() / 1e6:.4f} TWh, peak {y['peak']:.1f} MW, wind CF {y['wind'].mean():.3f}, solar CF {y['solar'].mean():.3f}")

assert len(Y["demand"]) == len(Y24["demand"]) == HOURS
assert abs(Y["peak"] - 450) < 1e-9 and abs(Y24["demand"].sum() - Y["demand"].sum()) < 1e-6
assert abs(Y["wind"].mean() - 0.338) < 1e-6 and abs(Y24["solar"].mean() - 0.240) < 1e-6
assert round(Y24["peak"]) == 462, "the lesson: at 2025's yearly energy, 2024's demand peaks at 462 MW"

## 2. Representative weeks and their weights

The year's first 52 weeks (8,736 hours) are the candidates. Each week belongs to the season of the month of its middle (winter is December to February).

- **Extremes** are taken first and kept as they were: the week of the year's highest hourly demand (`peak`), of the highest hourly net load, demand minus the output of a 150 MW wind farm and a 200 MW solar farm (`netpeak`), and of the least wind energy (`calm`).
- **Typical weeks**: within each season, the other weeks are split into $k$ groups (k-medoids). Each group's *medoid* is the real week with the smallest total distance to the others; the distance adds, hour by hour, the squared differences of demand (as a share of the peak), wind and solar per MW.
- **Scaling**: the medoid is scaled so that its group keeps its demand energy (wind and solar too, capped at the plant's size).
- **Weights**: one modeled hour of a week that stands for $n$ calendar weeks counts $w = n \times 8{,}760 / 8{,}736$ hours, which spreads the 24 hours of 31 December.

In [ ]:
SEASONS = [("winter", (12, 1, 2)), ("spring", (3, 4, 5)), ("summer", (6, 7, 8)), ("fall", (9, 10, 11))]
START = datetime(2025, 1, 1)   # 00:00 Central standard time, the first hour of town_year.json

def season_of(i):
    m = (START + timedelta(hours=i * HPW + 84)).month
    return next(name for name, months in SEASONS if m in months)

def select_weeks(Y, per_season=1, extremes=("peak", "netpeak", "calm"), net_ref=(150, 200)):
    d, w, s, P = Y["demand"], Y["wind"], Y["solar"], Y["peak"]
    wk = lambda a, i: a[i * HPW:(i + 1) * HPW]
    stats = [dict(d=wk(d, i).sum(), w=wk(w, i).sum(), s=wk(s, i).sum(), peak=wk(d, i).max(),
                  net=(wk(d, i) - net_ref[0] * wk(w, i) - net_ref[1] * wk(s, i)).max()) for i in range(WEEKS)]
    def dist(a, b):
        return float((((wk(d, a) - wk(d, b)) / P) ** 2 + (wk(w, a) - wk(w, b)) ** 2 + (wk(s, a) - wk(s, b)) ** 2).sum())
    taken, picks = set(), []
    def best(score, ok=lambda j: True):
        b = -1
        for j in range(WEEKS):
            if j not in taken and ok(j) and (b < 0 or score(j) > score(b)):
                b = j
        return b
    for kind in extremes:
        j = {"peak": lambda: best(lambda k: stats[k]["peak"]), "netpeak": lambda: best(lambda k: stats[k]["net"]),
             "calm": lambda: best(lambda k: -stats[k]["w"]),
             "winter": lambda: best(lambda k: stats[k]["net"], lambda k: season_of(k) == "winter")}[kind]()
        taken.add(j); picks.append((j, kind))
    def medoid(members):
        best_w, best_d = -1, float("inf")
        for c in members:
            tot = sum(dist(c, m) for m in members if m != c)
            if tot < best_d:
                best_d, best_w = tot, c
        return best_w
    def groups(members):
        if per_season == 1 or len(members) <= per_season:
            return [members]
        meds = [medoid(members)]
        while len(meds) < per_season:
            far, fd = -1, -1
            for m in members:
                if m in meds:
                    continue
                dd = min(dist(m, q) for q in meds)
                if dd > fd:
                    fd, far = dd, m
            meds.append(far)
        gs = None
        for _ in range(20):
            gs = [[] for _ in meds]
            for m in members:
                b, bd = 0, float("inf")
                for k, q in enumerate(meds):
                    dd = dist(m, q)
                    if dd < bd:
                        bd, b = dd, k
                gs[b].append(m)
            nm = [medoid(g) for g in gs]
            if nm == meds:
                return gs
            meds = nm
        return gs
    reps, chrono = [], [None] * WEEKS
    for name, _ in SEASONS:
        members = [j for j in range(WEEKS) if j not in taken and season_of(j) == name]
        for g in groups(members):
            b, n = medoid(g), len(g)
            sc = {k: sum(stats[m][k] for m in g) / n / stats[b][k] if stats[b][k] > 0 else 1 for k in ("d", "w", "s")}
            for m in g:
                chrono[m] = len(reps)
            reps.append(dict(week=b, kind="season", season=name, n=n, members=g, scale=sc))
    for j, kind in picks:
        chrono[j] = len(reps)
        reps.append(dict(week=j, kind=kind, season=season_of(j), n=1, members=[j], scale={"d": 1, "w": 1, "s": 1}))
    for r in reps:
        r["weight"] = r["n"] * HOURS / (WEEKS * HPW)
        a = r["week"] * HPW
        r["demand"] = d[a:a + HPW] * r["scale"]["d"]
        r["wind"] = np.minimum(1, w[a:a + HPW] * r["scale"]["w"])
        r["solar"] = np.minimum(1, s[a:a + HPW] * r["scale"]["s"])
    return reps, chrono

In [ ]:
WEEKS_OF = {}
for st in D["sets"]:
    reps, chrono = select_weeks(Y, st["perSeason"], st["extremes"])
    WEEKS_OF[st["id"]] = (reps, chrono)
    mine = [dict(week=r["week"], kind=r["kind"], season=r["season"], n=r["n"]) for r in reps]
    assert mine == REF["sets"][st["id"]]["picks"], f"{st['label']}: different weeks from the browser"
    assert abs(sum(r["weight"] * HPW for r in reps) - HOURS) < 1e-6 and None not in chrono
    print(f"{st['label']:30s} " + ", ".join(f"{r['kind'] if r['kind'] != 'season' else r['season']} wk{r['week']} x{r['n']}" for r in reps))
print("Every set of weeks matches the browser's.")

## 3. Costs

A plant built costs $f_i$ dollars per MW every year (battery energy: per MWh): overnight capital cost × capital recovery factor + fixed O&M.

- **Capital recovery factor**: the share of the capital cost paid each year to repay it with interest over the plant's life (4% real, 30 years; the battery 15).
- **Fixed O&M**: upkeep paid every year, whether the plant runs or not.
- **Running cost** of gas, per MWh made: heat rate × gas price + variable O&M. The heat rate is the gas burned per MWh, in MMBtu (one million British thermal units).
- **Constant output.** The nuclear plant makes 91.1% of its size in every hour.
- For a plant with constant output, fuel and variable O&M are a known yearly amount per MW built, so they are added to $f_i$.

In [ ]:
def crf(r, n):
    return 1 / n if r == 0 else r * (1 + r) ** n / ((1 + r) ** n - 1)

def costs(cfg, energy_capex=None):
    T, fin, gas = cfg["techs"], cfg["finance"], cfg["fuel"]["gas"]
    ef = cfg["fuel"]["co2PerMMBtu"] / 1000
    fixed = lambda capex, fom, life: 1000 * (capex * crf(fin["wacc"] / 100, life) + fom)
    C = {}
    for k in ("wind", "solar", "cc", "pk", "nuc"):
        t = T[k]
        C[k] = dict(fixed=fixed(t["capex"], t["fom"], t.get("life", fin["life"])),
                    run=t.get("heatRate", 0) * gas + t.get("fuel", 0) + t.get("vom", 0), co2=t.get("heatRate", 0) * ef)
    b = T["battery"]
    ec = b["energyCapex"] if energy_capex is None else energy_capex
    C["bp"] = dict(fixed=fixed(b["powerCapex"], b["powerCapex"] * b["fomShare"], b["life"]), run=0, co2=0)
    C["be"] = dict(fixed=fixed(ec, ec * b["fomShare"], b["life"]), run=0, co2=0)
    C["nuc"]["fixedTotal"] = C["nuc"]["fixed"] + C["nuc"]["run"] * T["nuc"]["avail"] * HOURS
    C["eta"], C["voll"], C["nucAvail"] = b["rte"], cfg["unservedPenalty"], T["nuc"]["avail"]
    return C

C = costs(cfg)
CW = costs(cfg, D["storage"]["energyCapex"])   # the what-if: storage energy at $20 per kWh (illustrative, not a cost forecast)
KEYS = ["wind", "solar", "cc", "pk", "nuc", "bp", "be"]
UNIT = {k: "MWh" if k == "be" else "MW" for k in KEYS}   # battery energy is sized in MWh, everything else in MW
print({k: f"${C[k]['fixed'] / 1e3:.1f}k per {UNIT[k]}-year" for k in KEYS})
print(f"gas combined-cycle: ${C['cc']['run']:.2f}/MWh, {C['cc']['co2']:.3f} t CO2/MWh; peaker ${C['pk']['run']:.2f}/MWh, {C['pk']['co2']:.3f} t/MWh")
assert round(C["wind"]["fixed"] / 1e3, 1) == 133.6 and round(C["cc"]["run"], 2) == 27.46 and round(C["cc"]["co2"], 3) == 0.329

## 4. The expansion LP

`linprog` solves: minimize $c^\top x$ subject to $A_{ub} x \le b_{ub}$, $A_{eq} x = b_{eq}$ and bounds. For each representative week $p$ and hour $h$:

| Rule | In symbols | Rows |
|---|---|---|
| Goal | $\min \sum_i f_i K_i + \sum_p w_p \sum_h (r_\text{cc} g_{p,h} + r_\text{pk} q_{p,h})$ | 1 |
| Demand | $a^w K_\text{wind} + a^s K_\text{solar} + 0.911 K_\text{nuc} + g + q + d - c \ge D$ | one per modeled hour |
| Sizes | $g \le K_\text{cc}$, $q \le K_\text{pk}$, $c + d \le K_\text{bp}$ | three per hour |
| Battery within week $p$ | $s_{p,h} = s_{p,h-1} + 0.85\,c - d$, $s_{p,0} = 0$, $\ell_p \le s_{p,h} \le u_p$ | three per hour |
| Link, calendar week $i$ | $S_{i+1} = S_i + s_{k(i),168}$ (the year closes), $S_i + \ell_{k(i)} \ge 0$, $S_i + u_{k(i)} \le K_\text{be}$ | three per calendar week |
| CO₂ | $\sum_p w_p \sum_h (0.329\,g + 0.484\,q) \le$ cap | 1 |
| Domains | $K, g, q, c, d, S \ge 0$; $s$ free in sign (measured from the week's start); $\ell_p \le 0 \le u_p$ | bounds, no rows |

Each step $h$ lasts one hour, so a power held for one step (MW) is that much energy (MWh). The class below collects rows as (row, column, value) triples and builds sparse matrices.

- `mode="cyclic"` drops the link equations and adds $s_{p,168} = 0$: each week ends where it starts, from a start level of its own.
- `mode="free"` drops the link equations only: each week may start at any level.

In [ ]:
class LP:
    def __init__(self):
        self.names, self.col, self.lb, self.ub, self.c = [], {}, [], [], []
        self.ub_rows, self.eq_rows = [], []      # (terms dict, rhs, name)
    def var(self, name, lb=0.0, ub=None, cost=0.0):
        self.col[name] = len(self.names); self.names.append(name); self.lb.append(lb); self.ub.append(ub); self.c.append(cost)
    def le(self, terms, rhs, name=None): self.ub_rows.append((terms, rhs, name))
    def ge(self, terms, rhs, name=None): self.ub_rows.append(({k: -v for k, v in terms.items()}, -rhs, name))
    def eq(self, terms, rhs, name=None): self.eq_rows.append((terms, rhs, name))
    def _mat(self, rows):
        r, c, v = [], [], []
        for i, (terms, _, _) in enumerate(rows):
            for k, x in terms.items():
                r.append(i); c.append(self.col[k]); v.append(x)
        return coo_matrix((v, (r, c)), shape=(len(rows), len(self.names))).tocsr(), np.array([b for _, b, _ in rows], float)
    def solve(self):
        A_ub, b_ub = self._mat(self.ub_rows)
        A_eq, b_eq = self._mat(self.eq_rows) if self.eq_rows else (None, None)
        t0 = time.perf_counter()
        res = linprog(np.array(self.c), A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=list(zip(self.lb, self.ub)), method="highs")
        assert res.status == 0, res.message
        res.x_ = dict(zip(self.names, res.x)); res.secs = time.perf_counter() - t0
        res.ub_names = [n for _, _, n in self.ub_rows]
        return res
    def size(self): return len(self.names), len(self.ub_rows) + len(self.eq_rows)

def week_lp(C, reps, chrono, cap, mode="linked"):
    m = LP()
    for k in KEYS:
        m.var("K_" + k, cost=C["nuc"]["fixedTotal"] if k == "nuc" else C[k]["fixed"])
    co2 = {}
    for p, r in enumerate(reps):
        w = r["weight"]
        m.var(f"lo{p}", None, 0.0); m.var(f"hi{p}", 0.0, None)
        for h in range(HPW):
            i = f"{p}_{h}"
            m.var("g" + i, cost=w * C["cc"]["run"]); m.var("q" + i, cost=w * C["pk"]["run"])
            m.var("c" + i); m.var("d" + i); m.var("s" + i, None, None)
            m.ge({"K_wind": r["wind"][h], "K_solar": r["solar"][h], "K_nuc": C["nucAvail"], "g" + i: 1, "q" + i: 1, "d" + i: 1, "c" + i: -1}, r["demand"][h])
            m.le({"g" + i: 1, "K_cc": -1}, 0); m.le({"q" + i: 1, "K_pk": -1}, 0); m.le({"c" + i: 1, "d" + i: 1, "K_bp": -1}, 0)
            dyn = {"s" + i: 1.0, "c" + i: -C["eta"], "d" + i: 1.0}
            if h > 0:
                dyn[f"s{p}_{h - 1}"] = -1.0
            m.eq(dyn, 0)
            m.ge({"s" + i: 1, f"lo{p}": -1}, 0); m.le({"s" + i: 1, f"hi{p}": -1}, 0)
            co2["g" + i] = w * C["cc"]["co2"]; co2["q" + i] = w * C["pk"]["co2"]
        if mode == "cyclic":
            m.eq({f"s{p}_{HPW - 1}": 1}, 0)
    for i in range(WEEKS):
        m.var(f"W{i}")
    for i in range(WEEKS):
        k = chrono[i]
        if mode == "linked":
            m.eq({f"W{(i + 1) % WEEKS}": 1, f"W{i}": -1, f"s{k}_{HPW - 1}": -1}, 0)
        m.ge({f"W{i}": 1, f"lo{k}": 1}, 0); m.le({f"W{i}": 1, f"hi{k}": 1, "K_be": -1}, 0)
    m.le(co2, cap, "co2")
    return m

def read_plan(C, res, reps):
    x = res.x_
    plan = {k: round(max(0.0, x["K_" + k]) * 1000) / 1000 for k in KEYS}
    co2 = sum(r["weight"] * (C["cc"]["co2"] * x[f"g{p}_{h}"] + C["pk"]["co2"] * x[f"q{p}_{h}"]) for p, r in enumerate(reps) for h in range(HPW))
    return dict(plan=plan, cost=res.fun, co2=co2)

In [ ]:
CAP = D["cap"] * 1e3
reps, chrono = WEEKS_OF["typ4"]
m = week_lp(C, reps, chrono, CAP)
res = m.solve()
est4 = read_plan(C, res, reps)
cols, rows = m.size()
print(f"4 typical weeks: {cols:,} columns x {rows:,} rows, solved in {res.secs:.2f} s")
print(f"estimate ${est4['cost'] / 1e6:.2f}M a year, CO2 {est4['co2'] / 1e3:.1f} kt;", {k: round(v, 1) for k, v in est4["plan"].items()})
ref = REF["sets"]["typ4"]
assert (cols, rows) == (ref["cols"], ref["rows"]), "the LP should have the browser's size"
assert abs(est4["cost"] / ref["cost"] - 1) < 1e-6, "the estimate should match the browser's"

## 5. The check: the plan on every hour of a year

Fix the sizes and run all 8,760 hours as one long period.

- **Rules:** the plan's own. Gas within its sizes; charge plus discharge within the battery's MW; its level within its MWh, cyclic over the year.
- **Unserved:** demand the plants cannot meet, at an illustrative \$10,000 per MWh. The weight keeps the lights on wherever possible; it is reported apart, never as a cost.
- **Spilled:** wind, sun and the flat nuclear output that nobody can use.

**Error measures** (the lesson's): cost gap = plan's spending on the full year ÷ weeks' estimate − 1, where spending counts plants and fuel only (no shortage penalty, and the re-run enforces no CO₂ cap); CO₂ against the cap; unserved energy in MWh, hours and the largest shortfall in MW.

You check **your own plan**: the cell first asserts that its sizes match the browser's to 0.01 MW (MWh for battery energy), then re-runs it and asserts spending, CO₂ and unserved energy.

In [ ]:
def check_lp(C, Y, plan):
    m, n = LP(), len(Y["demand"])
    bat = plan["bp"] > 0 and plan["be"] > 0
    for h in range(n):
        m.var(f"g{h}", 0, plan["cc"], C["cc"]["run"]); m.var(f"q{h}", 0, plan["pk"], C["pk"]["run"]); m.var(f"u{h}", cost=C["voll"])
        avail = Y["wind"][h] * plan["wind"] + Y["solar"][h] * plan["solar"] + C["nucAvail"] * plan["nuc"]
        bal = {f"g{h}": 1, f"q{h}": 1, f"u{h}": 1}
        if bat:
            m.var(f"c{h}", 0, plan["bp"]); m.var(f"d{h}", 0, plan["bp"]); m.var(f"s{h}", 0, plan["be"])
            bal[f"d{h}"] = 1; bal[f"c{h}"] = -1
            m.eq({f"s{h}": 1, f"c{h}": -C["eta"], f"d{h}": 1, f"s{(h - 1) % n}": -1}, 0)
            m.le({f"c{h}": 1, f"d{h}": 1}, plan["bp"])      # charge + discharge <= battery MW, the plan's own row
        m.ge(bal, Y["demand"][h] - avail)
    return m

def read_check(C, Y, plan, res):
    n = len(Y["demand"]); x = res.x_
    g = np.array([x[f"g{h}"] for h in range(n)]); q = np.array([x[f"q{h}"] for h in range(n)]); u = np.array([x[f"u{h}"] for h in range(n)])
    if "c0" in x:   # every hour: charge + discharge within the battery's MW
        assert max(x[f"c{h}"] + x[f"d{h}"] for h in range(n)) <= plan["bp"] + 1e-6, "the battery's power limit"
    fixed = sum(C[k]["fixed"] * plan[k] for k in KEYS)
    fuel = C["cc"]["run"] * g.sum() + C["pk"]["run"] * q.sum() + C["nuc"]["run"] * C["nucAvail"] * plan["nuc"] * n
    return dict(cost=fixed + fuel, co2=C["cc"]["co2"] * g.sum() + C["pk"]["co2"] * q.sum(), unserved=u.sum(),
                hours=int((u > 0.05).sum()), maxShort=u.max(), shortWeeks=sorted({min(WEEKS - 1, h // HPW) for h in np.nonzero(u > 0.05)[0]}))

def same(mine, ref, what):
    # asserted: spending to a millionth, CO2 to 1.5 t, unserved energy to 0.5 MWh. Printed, not asserted: hours short,
    # the weeks they fall in and the largest shortfall (MW). When several schedules of the same cost leave the same
    # unserved energy, another solver build may spread it over other hours.
    assert abs(mine["cost"] / ref["cost"] - 1) < 1e-6, f"{what}: cost"
    assert abs(mine["co2"] - ref["co2"]) < 1.5, f"{what}: CO2"
    assert abs(mine["unserved"] - ref["unserved"]) < 0.5, f"{what}: unserved energy"
    flag = "" if (mine["hours"], mine["shortWeeks"], round(mine["maxShort"], 1)) == (ref["hours"], ref["shortWeeks"], ref["maxShort"]) else "  (shortfall spread differently from the browser's; the energy agrees)"
    print(f"  {what}: ${mine['cost'] / 1e6:.2f}M, {mine['co2'] / 1e3:.1f} kt, {mine['unserved']:,.1f} MWh unserved in {mine['hours']} h, up to {mine['maxShort']:.1f} MW{flag}")

def same_plan(mine, ref, what):
    # sizes to 0.01 MW (battery energy: 0.01 MWh); another plan of the same cost would fail here, so say so
    dmw = max(abs(mine[k] - ref[k]) for k in KEYS if k != "be"); dmwh = abs(mine["be"] - ref["be"])
    assert dmw < 0.01 and dmwh < 0.01, f"{what}: your plan differs from the browser's by {dmw:.3f} MW and {dmwh:.3f} MWh (another optimum of the same cost?)"
    return dmw, dmwh

plan = est4["plan"]                       # your own plan, from the solve above
same_plan(plan, REF["sets"]["typ4"]["plan"], "4 typical weeks")
c25 = read_check(C, Y, plan, check_lp(C, Y, plan).solve())
same(c25, REF["sets"]["typ4"]["y2025"], "4 typical weeks on 2025")
print(f"  cost gap {c25['cost'] / REF['sets']['typ4']['cost'] - 1:+.1%}, CO2 against the cap {c25['co2'] / CAP - 1:+.1%}")

Now every set of weeks, on both years. The 11- and 15-week LPs take a few seconds each here, as they do in the browser.

In [ ]:
rows_out = []
for st in D["sets"]:
    reps, chrono = WEEKS_OF[st["id"]]
    ref = REF["sets"][st["id"]]
    m = week_lp(C, reps, chrono, CAP); res = m.solve(); est = read_plan(C, res, reps)
    assert m.size() == (ref["cols"], ref["rows"]) and abs(est["cost"] / ref["cost"] - 1) < 1e-6, st["label"]
    dmw, dmwh = same_plan(est["plan"], ref["plan"], st["label"])
    print(f"{st['label']}: estimate ${est['cost'] / 1e6:.2f}M ({res.secs:.1f} s); your plan differs from the browser's by {dmw:.3f} MW (power) and {dmwh:.3f} MWh (battery energy)")
    out = {}
    for name, YY in (("2025", Y), ("2024", Y24)):
        c = read_check(C, YY, est["plan"], check_lp(C, YY, est["plan"]).solve())   # your own plan, re-run
        same(c, ref["y" + name], f"on {name}")
        out[name] = c
    rows_out.append((st["label"], est["cost"], out["2025"], out["2024"]))

## 6. Storage between weeks: three rules

What-if: storage energy at \$20 per kWh instead of the sourced \$247 in 2025 dollars (illustrative), so that the solver builds a larger battery. Same seven weeks, same 200 kt cap.

Linked and cyclic estimates come out close, but that does not mean the linked battery stays within each week: the cell prints the largest net change over one representative week. Cyclic forces it to zero; linked only forces the calendar chain to close over the year.

In [ ]:
reps, chrono = WEEKS_OF[D["storage"]["set"]]
SCAP = D["storage"]["cap"] * 1e3
for mode in D["storage"]["modes"]:
    ref = REF["storage"][mode]
    res = week_lp(CW, reps, chrono, SCAP, mode).solve(); est = read_plan(CW, res, reps)
    net = sum(res.x_[f"s{chrono[i]}_{HPW - 1}"] for i in range(WEEKS))   # the calendar weeks' net changes, summed
    week_net = max(abs(res.x_[f"s{p}_{HPW - 1}"]) for p in range(len(reps)))   # the largest net change over one representative week
    assert abs(est["cost"] / ref["cost"] - 1) < 1e-6, mode
    same_plan(est["plan"], ref["plan"], mode)
    c = read_check(CW, Y, est["plan"], check_lp(CW, Y, est["plan"]).solve())   # your own plan, re-run
    assert abs(c["co2"] - ref["fullCo2"]) < 1.5 and abs(c["unserved"] - ref["full"]["unserved"]) < 0.5, mode
    real = week_lp(C, reps, chrono, SCAP, mode).solve().fun
    assert abs(real / ref["realCost"] - 1) < 1e-6, mode
    print(f"{mode:7s} battery {est['plan']['bp']:.0f} MW / {est['plan']['be']:,.0f} MWh, estimate ${est['cost'] / 1e6:.2f}M, largest net change in one week {week_net:,.0f} MWh, the year's {net:,.0f} MWh;"
          f" full year {c['co2'] / 1e3:.0f} kt, {c['unserved']:,.0f} MWh unserved; at the real cost ${real / 1e6:.2f}M")
    if mode == "free":
        assert net < -1000, "with no link, the battery ends the year with less than it began: energy from nothing"
    if mode == "cyclic":
        assert week_net < 1e-6, "cyclic: every representative week ends where it starts"

## 7. The full-year optimum and the estimate error

The same LP with all 8,760 hours as one period (no weights, the battery cyclic over the year): 43,807 columns. This is the reference for the **estimate error** = weeks' optimal cost ÷ full-year optimal cost − 1. It takes seconds to a minute.

In [ ]:
def full_lp(C, Y, cap):
    m, n = LP(), len(Y["demand"])
    for k in KEYS:
        m.var("K_" + k, cost=C["nuc"]["fixedTotal"] if k == "nuc" else C[k]["fixed"])
    co2 = {}
    for h in range(n):
        m.var(f"g{h}", cost=C["cc"]["run"]); m.var(f"q{h}", cost=C["pk"]["run"]); m.var(f"c{h}"); m.var(f"d{h}"); m.var(f"s{h}")
        m.ge({"K_wind": Y["wind"][h], "K_solar": Y["solar"][h], "K_nuc": C["nucAvail"], f"g{h}": 1, f"q{h}": 1, f"d{h}": 1, f"c{h}": -1}, Y["demand"][h])
        m.eq({f"s{h}": 1, f"c{h}": -C["eta"], f"d{h}": 1, f"s{(h - 1) % n}": -1}, 0)
        m.le({f"s{h}": 1, "K_be": -1}, 0); m.le({f"c{h}": 1, f"d{h}": 1, "K_bp": -1}, 0)
        m.le({f"g{h}": 1, "K_cc": -1}, 0); m.le({f"q{h}": 1, "K_pk": -1}, 0)
        co2[f"g{h}"] = C["cc"]["co2"]; co2[f"q{h}"] = C["pk"]["co2"]
    m.le(co2, cap, "co2")
    return m

m = full_lp(C, Y, CAP)
print("full-year LP:", m.size())
res = m.solve()
cap_dual = -res.ineqlin.marginals[res.ub_names.index("co2")]
F = REF["full"]
print(f"full-year optimum: ${res.fun / 1e6:.2f}M a year ({res.secs:.0f} s); the last tonne cut costs ${cap_dual:.1f}")
assert m.size() == (F["cols"], F["rows"]) and abs(res.fun / F["cost"] - 1) < 1e-6 and abs(cap_dual / F["capDual"] - 1) < 1e-3
for label, est_cost, c25, c24 in rows_out:
    print(f"  {label:30s} estimate error {est_cost / res.fun - 1:+.1%}")
fplan = {k: round(max(0.0, res.x_["K_" + k]) * 1000) / 1000 for k in KEYS}   # your own full-year plan
same_plan(fplan, F["plan"], "the full-year optimum")
f24 = read_check(C, Y24, fplan, check_lp(C, Y24, fplan).solve())
same(f24, F["y2024"], "the plan optimal on all of 2025, run on 2024")

## 8. What survives

| Conclusion from the seven weeks | Check | Verdict |
|---|---|---|
| The plan's spending on plants and fuel is within 2% of the estimate (no shortage penalty, no cap in the re-run) | both full years | holds |
| The plan serves every hour | both full years | fails |
| The plan meets the 200 kt cap | both full years | fails |
| Meeting 200 kt costs about \$129M | the full-year optimum | fails (−3.0%) |
| The town meets an adequacy criterion, such as a loss-of-load expectation of 0.1 days a year (PJM Manual 20A, §1.3, June 2026) | not tested: LOLE is an expected number of shortage days over many weather years with plant outages | open |

The cell below prints the numbers behind the table from your own runs.

In [ ]:
label, est7, c25, c24 = next(r for r in rows_out if r[0].startswith("4 typical + 3"))
print(f"cost gap: {c25['cost'] / est7 - 1:+.1%} (2025), {c24['cost'] / est7 - 1:+.1%} (2024): spending on plants and fuel only")
print(f"the shortage penalty left out: ${c25['unserved'] * C['voll'] / 1e6:.1f}M (2025), ${c24['unserved'] * C['voll'] / 1e6:.1f}M (2024)")
print(f"unserved: {c25['unserved']:,.0f} MWh (2025), {c24['unserved']:,.0f} MWh (2024)")
print(f"CO2: {c25['co2'] / 1e3:.0f} kt (2025), {c24['co2'] / 1e3:.0f} kt (2024), cap {CAP / 1e3:.0f} kt")
print(f"estimate error against the full-year optimum: {est7 / res.fun - 1:+.1%}")
assert abs(c25["cost"] / est7 - 1) <= 0.02 and abs(c24["cost"] / est7 - 1) <= 0.02 and c25["unserved"] > 0 and c24["unserved"] > 0 and c25["co2"] > CAP
print("The lab matches the browser.")

## 9. Practice (about 120 minutes)

Write your own cells below each task. Predict first, then solve, and state each result with an error measure.

1. **A fourth extreme** (about 20 minutes). Add the winter week with the highest net load, chosen after the other three (`extremes=("peak", "netpeak", "calm", "winter")`). Does the full-year shortfall on 2025 and on 2024 fall? What does the extra week cost in LP size?
2. **Days instead of weeks** (about 30 minutes).
   - Rewrite `select_weeks` for 24-hour periods: 12 typical days per season and the three extreme days.
   - `week_lp` assumes 168-hour periods and 52 calendar weeks (`HPW`, `WEEKS`). Give it the period length and `len(chrono)` (365 days) instead, so the battery is linked across the days.
   - Compare the estimate error and the full-year shortfall with the weeks'. Which keeps multi-day calm spells?
3. **Seasonal storage** (about 20 minutes).
   - With storage energy at \$20 per kWh (the illustrative what-if) and a 100 kt cap, compare linked and cyclic.
   - Find a representative week whose linked net change (ending level minus starting level, `s{p}_167`) is not zero. Why does the cyclic rule require it to be zero?
4. **Meet the cap on the full year** (about 20 minutes). Lower the cap in the seven-week LP step by step until the plan's full-year CO₂ on 2025 is at most 200 kt. What does that cost, and does the plan then meet 200 kt on 2024?
5. **Another error measure** (about 15 minutes).
   - Net load: demand minus the output of the reference farms that pick the net-load week (`net_ref`: 150 MW of wind, 200 MW of solar), the same for every set.
   - Full year: sort its 8,760 hourly values, highest first.
   - Each set of weeks: give each modeled hour its weight in hours, sort, and read the weighted curve at the same 8,760 positions.
   - Error: the root-mean-square difference, in MW. Does it rank the five sets as the estimate error does?
6. **Write it up** (about 15 minutes). In five sentences, report the seven-week plan as a researcher would: the chronology specification (weights, the link, the extremes, the weather year), the two error measures with numbers, and what the runs cannot show.